# 01 — Régressions linéaires

## Rôle dans le projet
On prédit le **débit de chaque capteur 15 min plus tard (t+1)**. Ce notebook pose les **références linéaires** : on compare deux façons de répartir les coefficients d'une régression linéaire entre les 36 capteurs, sans aucune pénalisation. L'objectif est de comprendre **ce que gagne un modèle à être spécifique à chaque capteur** et **à partir de quand il surapprend**.

## Les modèles
Notons $x_i(t) \in \mathbb{R}^{48}$ les 48 variables du capteur $i$ et $x(t) \in \mathbb{R}^{1728}$ l'ensemble des variables des 36 capteurs.

**Global** : une seule fonction linéaire relie toutes les variables de tous les capteurs à tous les flux.

$$\hat y(t+1) = W\,x(t) + b, \qquad W \in \mathbb{R}^{36 \times 1728}$$

**Local** : un modèle par capteur, qui n'utilise que ses propres variables.

$$\hat y_i(t+1) = \mu_i + \beta_i^\top x_i(t)$$

Tous les coefficients sont appris par **moindres carrés ordinaires** (`LinearRegression`).


## Protocole
- Mêmes jeux train / validation / test que `02_mlp.ipynb`, avec un décalage d'un pas de temps entre `X` et `Y` : `X[1:]` et `Y[:-1]`, appliqué avant le découpage.
- Les modèles sont comparés sur la validation, le test n'est regardé qu'à la fin.
- Métriques : RMSE et MAE, en débit normalisé.
- Référence à battre : la **persistance** (le dernier débit connu).

Pas de `StandardScaler` : les débits sont déjà compris entre 0 et 1.



## Préparation
Imports, puis chargement des données sous forme de tableaux (T, 36, 48).

In [10]:
import sys; sys.path.append("..")
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from src.data import load_raw, train_val_split, DATA_PATH, T1_PATH
from src.make_dataset import main as make_dataset
from src.utils import evaluate
from src import var_graphe
from sklearn.linear_model import LinearRegression

BLEU, ORANGE, VERT, JAUNE, GRIS = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#898781"

if not T1_PATH.exists():  
    make_dataset()

X_tr, Y_tr, X_te, Y_te, A = load_raw(T1_PATH)
(X_a, Y_a), (X_v, Y_v) = train_val_split(X_tr, Y_tr)

print("train", X_a.shape, "| validation", X_v.shape, "| test", X_te.shape)
print ("y train",Y_a.shape,"| y_val",Y_v.shape,"| y_test",Y_te.shape)
print("train", X_a.shape, "| validation", X_v.shape, "| test", X_te.shape)
print ("y train",Y_a.shape,"| y_val",Y_v.shape,"| y_test",Y_te.shape)



train (1008, 36, 48) | validation (253, 36, 48) | test (840, 36, 48)
y train (1008, 36) | y_val (253, 36) | y_test (840, 36)
train (1008, 36, 48) | validation (253, 36, 48) | test (840, 36, 48)
y train (1008, 36) | y_val (253, 36) | y_test (840, 36)


## Fonctions utilitaires

- `scores` compare deux tableaux (T, 36) et renvoie le MAE et le RMSE.
- `persistance` renvoie le dernier débit connu, la colonne 9 de X. C'est la référence à battre.
- `courbes` trace un graphique en lignes. Les graphiques sont interactifs : survole-les pour lire les valeurs.
- `modele_global` ajuste une seule régression linéaire qui relie les 1728 variables des 36 capteurs aux 36 débits.
- `modele_local` ajuste une régression par capteur, qui n'utilise que ses propres 48 variables.

Les deux modèles prennent `X_fit`, `Y_fit` (pour apprendre) et `X_pred` (pour prédire), et renvoient un tableau (T, 36) que l'on passe à `scores`. Ils s'ajustent sans pénalisation, donc sans réglage d'hyperparamètre : on les compare directement sur la validation, puis sur le train pour repérer le surapprentissage.

In [3]:
def persistance(X):
    """sors le modèle naif"""
    return X[:, :, 9]

def scores(Y, Y_pred):
    """return le MEA et RMSE du modele"""
    return evaluate(Y.ravel(), Y_pred.ravel())


print("persistance (val)", scores(Y_v, persistance(X_v)))

persistance (val) {'MAE': 0.025418882281786657, 'RMSE': 0.03809453642424843}


## Analyse du modèle global

In [ ]:

def modele_global(X_fit, Y_fit, X_pred):
    """UNE fonction linéaire : toutes les variables de tous les capteurs -> tous les capteurs."""
    F_fit = X_fit.reshape(len(X_fit), -1)    # (n, 1728)
    F_pred = X_pred.reshape(len(X_pred), -1)
    return LinearRegression().fit(F_fit, Y_fit).predict(F_pred)   # (n_pred, 36)

Y_pred_pers = persistance(X_v)
Y_pred_global = modele_global(X_a, Y_a, X_v)
Y_pred_global_train = modele_global(X_a, Y_a, X_a)

print("global      (train)", scores(Y_a, Y_pred_global_train))
print("global      (val)  ", scores(Y_v, Y_pred_global))
print("persistance (val)  ", scores(Y_v, Y_pred_pers))


#analyse des résultats

def comparer(predictions, Y, titre="Comparaison des modèles (validation)"):
    """Barres RMSE et MAE pour plusieurs modèles.

    predictions : dict {nom du modèle: prédictions (n, 36)}
    Y           : valeurs réelles (n, 36)
    """
    noms = list(predictions)
    s = {nom: scores(Y, P) for nom, P in predictions.items()}
    fig = go.Figure([
        go.Bar(name="RMSE", x=noms, y=[s[n]["RMSE"] for n in noms], marker_color=BLEU,text=[f"{s[n]['RMSE']:.4f}" for n in noms], textposition="outside"),
        go.Bar(name="MAE", x=noms, y=[s[n]["MAE"] for n in noms], marker_color=ORANGE,text=[f"{s[n]['MAE']:.4f}" for n in noms], textposition="outside"),
    ])
    fig.update_layout(template="plotly_white", barmode="group", title=titre, yaxis_title="erreur")
    return fig

comparer({"Persistance": persistance(X_v),"Global": Y_pred_global}, Y_v).show()

global      (train) {'MAE': 0.014911070990268645, 'RMSE': 0.02147138464423478}
global      (val)   {'MAE': 0.02710062377174, 'RMSE': 0.039805056616234694}
persistance (val)   {'MAE': 0.025418882281786657, 'RMSE': 0.03809453642424843}


Le modèle global a trop de coefficients pour le nombre d'échantillons, et on se retrouve face à un problème d'over-fitting qui rend le résultat moins bon que celui du modèle naif. 

On analyse maintenant un modèle de regression linéaire local.

## Modèle local



Au lieu d'un seul modèle qui lit tous les capteurs, on apprend **un modèle par capteur**, qui n'utilise que ses propres 48 variables : on passe de 1728 à 48 coefficients par modèle, soit bien moins que les 1008 échantillons.

Ce choix a un coût : chaque capteur ne voit plus que lui-même et ignore l'information de ses voisins. On compare ce modèle au global sur la validation pour voir si cette simplification est gagnante.

In [15]:
def modele_local(X_fit, Y_fit, X_pred):
    """UNE fonction linéaire par capteur (36 modèles indépendants)."""
    Y_pred = np.zeros((len(X_pred), Y_fit.shape[1]))
    for i in range(Y_fit.shape[1]):
        Y_pred[:, i] = LinearRegression().fit(X_fit[:, i, :], Y_fit[:, i]).predict(X_pred[:, i, :])
    return Y_pred

Y_pred_local = modele_local(X_a, Y_a, X_v)
print("local       (val)", scores(Y_v, Y_pred_local))


Y_pred_pers = persistance(X_v)
s_pers, s_local = scores(Y_v, Y_pred_pers), scores(Y_v, Y_pred_local)

fig = go.Figure([
    go.Bar(name="Persistance", x=["RMSE", "MAE"], y=[s_pers["RMSE"], s_pers["MAE"]], marker_color=GRIS,
           text=[f"{s_pers['RMSE']:.4f}", f"{s_pers['MAE']:.4f}"], textposition="outside"),
    go.Bar(name="Local", x=["RMSE", "MAE"], y=[s_local["RMSE"], s_local["MAE"]], marker_color=VERT,
           text=[f"{s_local['RMSE']:.4f}", f"{s_local['MAE']:.4f}"], textposition="outside"),
])
fig.update_layout(template="plotly_white", barmode="group",
                  title="Validation : modèle local contre persistance", yaxis_title="erreur")
fig.show()

gain = 1 - s_local["RMSE"] / s_pers["RMSE"]
print(f"gain du local sur la persistance (RMSE) : {gain:.1%}")

local       (val) {'MAE': 0.022051863187443718, 'RMSE': 0.03385006972393662}


gain du local sur la persistance (RMSE) : 11.1%


Le modèle local fait mieux que la persistance, avec un RMSE de validation de 0,0339 contre **[RMSE persistance]** (soit **[gain] %** de moins), et mieux que le modèle global (0,0398) : réduire le nombre de coefficients aide. Mais chaque capteur ne lit que lui-même, alors que les débits d'un réseau routier sont liés entre voisins.

## Prise en compte des voisins

On élargit donc l'entrée de chaque modèle local aux variables des capteurs situés à au plus $K$ sauts dans le graphe ($K = 0$ redonne le modèle local, $K = 1$ ajoute les voisins directs, $K = 2$ les voisins à deux sauts). Il faut trouver le bon compromis : plus de voisins apportent de l'information, mais aussi plus de coefficients à apprendre avec les mêmes 1008 échantillons.


In [18]:
def masque_sauts(A, sauts):
    """True si le capteur j est à au plus `sauts` sauts du capteur i (lui-même inclus)."""
    pas = ((A > 0) | np.eye(len(A), dtype=bool)).astype(int)
    return np.linalg.matrix_power(pas, sauts) > 0


def modele_local_voisins(X_fit, Y_fit, X_pred, A, sauts=1):
    """UNE fonction linéaire par capteur, sur ses variables ET celles de ses voisins jusqu'à `sauts` sauts."""
    n_cap = Y_fit.shape[1]
    masque = masque_sauts(A, sauts)
    Y_pred = np.zeros((len(X_pred), n_cap))
    for i in range(n_cap):
        voisins = np.flatnonzero(masque[i])
        F_fit = X_fit[:, voisins, :].reshape(len(X_fit), -1)
        F_pred = X_pred[:, voisins, :].reshape(len(X_pred), -1)
        Y_pred[:, i] = LinearRegression().fit(F_fit, Y_fit[:, i]).predict(F_pred)
    return Y_pred


Y_pred_local_voisins1 = modele_local_voisins(X_a, Y_a, X_v, A, sauts=1)
Y_pred_local_voisins2 = modele_local_voisins(X_a, Y_a, X_v, A, sauts=2)


# print("train", X_a.shape, "| validation", X_v.shape)
# print ("y train",Y_a.shape,"| y_pred_global",Y_pred_global.shape,"| y_pred_local",Y_pred_local.shape)


print("local voisins 1(val)", scores(Y_v, Y_pred_local_voisins1))
print("local voisins 2(val)", scores(Y_v, Y_pred_local_voisins2))



resultats = {
    "Persistance": scores(Y_v, persistance(X_v))["RMSE"],
    "K = 0 (local)": scores(Y_v, Y_pred_local)["RMSE"],
    "K = 1": scores(Y_v, Y_pred_local_voisins1)["RMSE"],
    "K = 2": scores(Y_v, Y_pred_local_voisins2)["RMSE"],
}

fig = go.Figure(go.Bar(
    x=list(resultats), y=list(resultats.values()),
    marker_color=[GRIS, BLEU, VERT, JAUNE],
    text=[f"{v:.4f}" for v in resultats.values()], textposition="outside",
))
fig.update_layout(template="plotly_white", title="RMSE de validation selon le voisinage",
                  yaxis_title="RMSE", showlegend=False)
fig.show()

local voisins 1(val) {'MAE': 0.019974685872313904, 'RMSE': 0.030740418816478647}
local voisins 2(val) {'MAE': 0.020505103216793493, 'RMSE': 0.031161586564498286}


## Conclusion

Les voisins améliorent le modèle local : le RMSE de validation passe de 0,0339 (K = 0) à **0,0307** avec les voisins directs (K = 1), soit environ 9 % de moins. Dans ce contexte, **K = 1 est le meilleur modèle** : les voisins proches apportent de l'information utile, mais en ajouter davantage (K = 2, ou tous les capteurs comme dans le modèle global, à 0,0398) multiplie les coefficients et fait surapprendre, car les 1008 échantillons d'entraînement ne suffisent plus.

Ce compromis est la limite des moindres carrés ordinaires : **à chaque saut ajouté, le nombre de coefficients augmente sans aucun garde-fou**, et le surapprentissage menace dès que le voisinage grandit. Rien ne permet de garder l'information des capteurs lointains sans payer ce coût.



Dans la prochaine partie, nous allons garder la structure « un modèle par capteur qui lit ses voisins », mais on **pénalise les grands coefficients** (régression Ridge) pour contrôler la complexité, ce qui permet d'élargir le voisinage sans surapprendre. On ajoute aussi le **calendrier** (jour et heure) comme entrée. Le nombre de sauts $K$ et la pénalisation $\alpha$ sont choisis sur la validation : c'est l'objet de `03_var_graphe.ipynb`.